# Sample design and GRAME product ingestion

MODCiX reloaded pipeline, stages 1-3 (see repo README for the full pipeline):

1. Simple random sample of 2x2km grid cells over Europe, stratified visually by mowing-intensity/bioregion coverage.
2. Ingestion of the official GRAME product (via WEkEO) as a GEE asset, used both for stratifying the pixel-level sample (`gee_scripts/01_stratified_pixel_sampling.js`) and as the map being validated.

PlanetScope scene search, ordering, and GEE ingestion (previously inline here) has been extracted into
[`scripts/planetscope_to_gee_ingestion.py`](../scripts/planetscope_to_gee_ingestion.py) - see the markdown cell
further down for how the two connect.

Secrets (Planet API key, WEkEO credentials, GEE project/bucket) are loaded from `config/catalog.yaml` via
`config/.env` - copy `config/template.env` to `config/.env` and fill in your own values before running this
notebook (see README.md).

### Setup environment

In [ ]:
!pip install duckdb

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from src.config import load_catalog
catalog = load_catalog()

# Root folder for this project's geospatial data (see config/template.env -> PROJDIR).
# Not tracked in git - large geospatial inputs/outputs live outside the repo. See README.md.
projDir = catalog['project_data']['filepath']

In [ ]:
!ls {projDir + '/data'}

### Bring in GRAME & bioregion data from Julien - aggregaetd to 2km grid

In [ ]:
import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt

# path to your file
db_path = projDir + '/data/cdiv_hrl_eea38_grame.duckdb'

# Path to 2km grid
grid_path = projDir + "/data/grid_2km_surf.gpkg"

con = duckdb.connect(db_path)
con.install_extension("spatial")
con.load_extension("spatial")

In [ ]:
con.sql("SHOW TABLES").df()

In [ ]:
con.sql("SELECT * FROM zon_hist_2km_2021_melted_bioreg LIMIT 10").df()

In [ ]:
con.sql(""" SELECT DISTINCT crop FROM zon_hist_2km_2023_melted_bioreg """).df()

### Calculate a mowing intensity index and grassland fraction

In [ ]:
mow_intensity = con.sql("""
WITH recoded AS (
    SELECT
        GRD_ID,
        code,
        hist,
        CASE
            WHEN CAST(crop AS INTEGER) IN (0,1,2,3,4) THEN CAST(crop AS INTEGER)
            ELSE NULL
        END AS mow_count
    FROM zon_hist_2km_2023_melted_bioreg
    WHERE CAST(crop AS INTEGER) IN (0,1,2,3,4,253)
)

SELECT
    GRD_ID,
    ANY_VALUE(code) AS code,

    -- only grassland pixels
    SUM(CASE WHEN mow_count IS NOT NULL THEN hist END) AS total_grass_pixels,

    -- weighted sum of mowing events
    SUM(hist * mow_count) * 1.0 
        / SUM(CASE WHEN mow_count IS NOT NULL THEN hist END) 
        AS mowing_intensity,
    
    -- grassland fraction
    SUM(CASE WHEN mow_count IS NOT NULL THEN hist END) * 1.0 / SUM(hist)
    AS grassland_fraction

FROM recoded
GROUP BY GRD_ID
""").df()

In [ ]:
mow_intensity

In [ ]:
mow_intensity.isna().sum()

In [ ]:
# 2) Read the 2 km grid as a GeoDataFrame
grid_gdf = gpd.read_file(grid_path)

# 3) Join mowing intensity to the grid
grid_mow = grid_gdf.merge(mow_intensity, on="GRD_ID", how="left")

grid_mow.head()

In [ ]:
grid_mow_clean = grid_mow.dropna(subset=["mowing_intensity"])

In [ ]:
len(grid_mow_clean) / len(grid_gdf)  * 100

In [ ]:
len(grid_mow_clean) == len(mow_intensity)

In [ ]:
# 4) Plot histogram of mowing intensity
plt.figure(figsize=(8, 5))
plt.hist(grid_mow_clean["mowing_intensity"], bins=30)
plt.xlabel("Mowing intensity index")
plt.ylabel("Number of 2 km grid cells")
plt.title("Distribution of mowing intensity index")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(15, 10))

grid_mow_clean.plot(
    column="mowing_intensity",
    ax=ax,
    legend=True,
    cmap="viridis",
    linewidth=0
)

ax.set_title("Mowing Intensity Index (2 km grid)")
ax.set_axis_off()

plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 10))

grid_mow_clean.plot(
    column="grassland_fraction",
    ax=ax,
    legend=True,
    cmap="viridis",
    linewidth=0
)

ax.set_title("Grassland fraction (2 km grid)")
ax.set_axis_off()

plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 10))

grid_mow_clean.plot(
    column="code",
    ax=ax,
    legend=True,
    linewidth=0
)

ax.set_title("Bioregion (2 km grid)")
ax.set_axis_off()

plt.show()

### Generate simple random sample - covers grazing intensity gradient and bioregions well

In [ ]:
import pandas as pd

sample_df = grid_mow_clean.copy()

sample_df["mow_class"] = pd.qcut(
    sample_df["mowing_intensity"],
    q=4,
    labels=["low", "medium", "high", "very_high"],
    duplicates="drop"
)

# Filter out Macaronesia and Arctic
sample_df = sample_df[~sample_df["code"].isin(["Macaronesia", "Arctic"])]

In [ ]:
stratum_counts = (
    sample_df.groupby(["code", "mow_class"])
    .size()
    .reset_index(name="n")
)

stratum_counts.sort_values(["code", "mow_class"]).head(50)

In [ ]:


#n_per_stratum = 10

# Stratified sample
#sample_strat = (
#    sample_df.groupby([ "mow_class"], group_keys=False)
#    .apply(lambda x: x.sample(n=min(len(x), n_per_stratum), random_state=42))
#    .reset_index(drop=True)
#)

# Simple random sample
sample_strat = sample_df.sample(n=50, random_state=13)
sample_strat = sample_strat[["GRD_ID", "mow_class", "code", "grassland_fraction", "mowing_intensity"]]

In [ ]:
# Marcel wanted to add some grid cells to the sample
target_grd_ids = [
    "CRS3035RES2000mN2768000E4464000",
    "CRS3035RES2000mN2756000E4474000",
    "CRS3035RES2000mN2756000E4404000",
    "CRS3035RES2000mN2768000E4468000",
    "CRS3035RES2000mN2746000E4332000",
    "CRS3035RES2000mN2778000E4230000",
    "CRS3035RES2000mN3338000E4240000",
    "CRS3035RES2000mN3392000E4180000",
    "CRS3035RES2000mN3394000E4280000",
    "CRS3035RES2000mN3246000E4472000"
]

# 2. Filter the DataFrame
sample_strat_marcel = sample_df[sample_df['GRD_ID'].isin(target_grd_ids)]
sample_strat_marcel = sample_strat_marcel[["GRD_ID", "mow_class", "code", "grassland_fraction", "mowing_intensity"]]

In [ ]:
sample_strat_counts = (
    sample_strat.groupby(["code"])
    .size()
    .reset_index(name="n")
)

sample_strat_counts.sort_values(["code"]).head(50)

In [ ]:
sample_strat_counts = (
    sample_strat.groupby(["mow_class"])
    .size()
    .reset_index(name="n")
)

sample_strat_counts.sort_values([ "mow_class"]).head(50)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(sample_strat["grassland_fraction"], bins=10)
plt.xlabel("Grassland fractioN")
plt.ylabel("Number of 2 km grid cells in sample")
plt.title("Distribution of grassland fraction")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(sample_strat["mowing_intensity"], bins=10)
plt.xlabel("Mowing intensity")
plt.ylabel("Number of 2 km grid cells in sample")
plt.title("Distribution of mowing intensity")
plt.show()

In [ ]:
sample_strat

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# 1) Keep only the sampled rows from the original GeoDataFrame
sample_map = grid_mow_clean.merge(
    sample_strat[["GRD_ID", "mow_class", "code"]],
    on=["GRD_ID", "code"],
    how="inner"
).copy()


# 2) Make sure CRS is carried over from grid_mow_clean
sample_map = sample_map.set_crs(grid_mow_clean.crs, allow_override=True)

# 3) Convert grid polygons to centroids for easier plotting
sample_pts = sample_map.copy()
sample_pts["geometry"] = sample_pts.geometry.centroid


countries = gpd.read_file(projDir + "/data/countries.geojson")
# 5) Reproject country boundaries to the same CRS as the sample/grid
countries = countries.to_crs(sample_pts.crs)

# Optional: crop countries to the sample extent for faster plotting
xmin, ymin, xmax, ymax = sample_pts.total_bounds
countries_clip = countries.cx[xmin:xmax, ymin:ymax]

# ----------------------------
# Map 1: biome code
# ----------------------------
fig, ax = plt.subplots(figsize=(15, 10))

countries_clip.boundary.plot(ax=ax, linewidth=0.5, color="lightgrey")

sample_pts.plot(
    column="code",
    ax=ax,
    legend=True,
    markersize=12
)
# Crop to extent
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.set_title("Sampled grid-cell centroids by biome")
ax.set_axis_off()

plt.show()

# ----------------------------
# Map 2: mowing intensity class
# ----------------------------
fig, ax = plt.subplots(figsize=(15, 10))

countries_clip.boundary.plot(ax=ax, linewidth=0.5, color="lightgrey")

sample_pts.plot(
    column="mow_class",
    ax=ax,
    legend=True,
    markersize=12,
    cmap="viridis"
)
# Crop to extent
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.set_title("Sampled grid-cell centroids by mowing intensity category")
ax.set_axis_off()

plt.show()

### PlanetScope ingestion (extracted to a standalone script)

The sample grid cells selected above were exported (as `sample_grids_batch1.geojson`, `sample_grids_batch2.geojson`,
plus a small hand-picked `sample_grids_batch_marcel.geojson`) and handed to
[`scripts/planetscope_to_gee_ingestion.py`](../scripts/planetscope_to_gee_ingestion.py), which:

1. Searches the Planet Data API per grid cell for PSScene imagery (8-band SR, cloud cover <= 30%) overlapping
   the cell by more than 25%.
2. Submits Planet Orders API v2 orders delivering matched scenes directly into a staging GEE collection.
3. Polls each order and relays (renames) the delivered assets from staging into the permanent target GEE
   collection (`gee_scripts/01_stratified_pixel_sampling.js` and `gee_scripts/02_sampling_app.js` both read from
   this permanent collection).

```bash
python scripts/planetscope_to_gee_ingestion.py \
    --grid-files data/for_gee/sample_grids_batch1.geojson data/for_gee/sample_grids_batch2.geojson data/for_gee/sample_grids_batch_marcel.geojson \
    --start-date 2021-01-01 --end-date 2021-12-31 \
    --target-collection PlanetScope/Europe_2kmGridSample_ortho_analytic_8b_sr \
    --staging-collection test_planetscope
```

The merged/finalised sample grid (all batches, exported for GEE) is produced by the last cell of this notebook.

### Querying WEKEO API to get GRAME rasters for upload to GEE

In [ ]:
grid_gdf_wgs84 = grid_gdf.to_crs(epsg=4326)
grid_gdf_wgs84.total_bounds
bbox = dict(zip(["minx","miny","maxx","maxy"], grid_gdf_wgs84.total_bounds))
bbox

In [ ]:

from hda import Client, Configuration
from pathlib import Path
import getpass
import json

# ----------------------------
# 1) Authenticate
# ----------------------------
# Option A: direct credentials
conf = Configuration(user=catalog['wekeo_account']['user'], password=catalog['wekeo_account']['password'])
hda_client = Client(config=conf)



In [ ]:

# ----------------------------
# 2) Define your query
# ----------------------------
query = {
    "dataset_id": "EO:EEA:DAT:HRL:GRA",
    "bbox": [-63.18000000000001, -21.424, 55.8535, 71.2],
    "productType": "Grassland Mowing Events",
    "resolution": "10m",
    "year": "2021",
    "itemsPerPage": 200,
    "startIndex": 0
}

print(json.dumps(query, indent=2))

# ----------------------------
# 3) Search
# ----------------------------
matches = hda_client.search(query)

print(matches)          # summary
print(len(matches))     # number of matched items, if supported

# Inspect a few result IDs / filenames if available
for i, item in enumerate(matches[:5]):
    print(f"\nResult {i+1}")
    print(item)

In [ ]:
download_dir = projDir + "/data/for_gee/wekeo_grassland_mowing_2021"
Path(download_dir).mkdir(parents=True, exist_ok=True)

#matches.download(download_dir=download_dir)

In [ ]:
zip_files = !ls $download_dir
len(zip_files)

In [ ]:
from pathlib import Path

tif_dir = Path(download_dir) / "geotiffs"
tif_dir.mkdir(exist_ok=True)

In [ ]:
import zipfile
from pathlib import Path

zip_files = Path(download_dir).glob("*.zip")

for zfile in zip_files:
    with zipfile.ZipFile(zfile, "r") as zip_ref:
        for member in zip_ref.namelist():
            if member.endswith(".tif"):
                zip_ref.extract(member, tif_dir)

In [ ]:
import ee
project_id = catalog['gee_project']['name']
ee.Initialize(project=project_id)
print(ee.String('Hello from the Earth Engine servers!').getInfo())

In [ ]:
project_id = catalog['gee_project']['name']
bucket_name = catalog['gee_project']['bucket']
!gcloud config set project {project_id}
!gsutil ls

In [ ]:
!ls $tif_dir

In [ ]:
from pathlib import Path

filtered_files = !ls $tif_dir

total_files = len(filtered_files)

for x in range(total_files):
    localFileDir = str(Path(tif_dir) / filtered_files[x])
    
    progress = (x + 1) / total_files * 100
    print(f"Processing file {x + 1}/{total_files} ({progress:.2f}%)")
    print(f"Uploading: {localFileDir}")
    
    !gsutil -m cp "{localFileDir}" gs://{bucket_name}/

In [ ]:
gcsFiles = !gsutil ls gs://{bucket_name}
gcsFiles
len(gcsFiles)

In [ ]:
gcsFiles

In [ ]:
# Total number of files for progress calculation
total_files = len(gcsFiles)

for x in range(total_files):
    fileName = gcsFiles[x]
    imgName = fileName.split('.')[0].split('/')[3]
    
    # Calculate and print progress
    progress = (x + 1) / total_files * 100
    print(f"Processing file {x + 1}/{total_files} ({progress:.2f}%)")
    
    # Print details of the file being processed
    print(f"Uploading: {fileName} (imgName: {imgName})")
    
    # Run the Earth Engine upload command
    !earthengine --project {project_id} upload image \
    --asset_id=projects/{project_id}/Europe_misc/CLMS_HRLVLCC_GRAME_S2021_R10m/$imgName \
    $fileName

In [ ]:
# Recursively delete all files in cloud bucket
!gsutil rm -r gs://{bucket_name}/*

In [ ]:
import geopandas as gpd
import pandas as pd

# Define your file paths
files = [
    projDir + "/data/sample_grids_batch1.geojson",
    projDir + "/data/sample_grids_batch2.geojson"
]

# 1. Read all files into a list of GeoDataFrames
gdf_list = [gpd.read_file(f) for f in files]

# 2. Concatenate them into one GeoDataFrame
merged_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))

# 3. Export to Shapefile
output_path = projDir + "/data/for_gee/modcix_planetscope_grid_samples.shp"
merged_gdf.to_file(output_path, driver='ESRI Shapefile')

print(f"Successfully exported merged data to: {output_path}")